# Qwen-14B Instruct Kaggle Runner (CSV Pipeline)

This notebook runs local Qwen inference on Kaggle and writes results to CSV in the same schema as your current experiment pipeline.

## What you upload to Kaggle input
- Qwen model directory (for example: `/kaggle/input/qwen2.5/transformers/14b-instruct/1`)
- Prompt file (`EVIDENCE_PROMPT.md` or your custom prompt)
- Questions file (`.jsonl` with `id`, `table_id`, `query`, `label`)
- Table directory (`table_inputs_2` for `txt`, or your `csv/json/html` table files)

## Execution order
1. Update path values in `CFG` in the next code cell.
2. Run the setup/helpers cell.
3. Run the byte-identical override cell.
4. Run model loading cell.
5. Run prompt preview cell.
6. Run full experiment cell.

In [ ]:
import csv
import json
import re
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Raise CSV field size limit — C5's 6-step responses can exceed the 131 KB default
csv.field_size_limit(sys.maxsize)
PROJECT_ROOT = Path.cwd()


@dataclass
class RunConfig:
    # Update these paths to match your Kaggle input datasets.
    model_name: str = "/kaggle/input/qwen2.5/transformers/14b-instruct/1"
    prompt_file: str = "/kaggle/input/your-input-dataset/EVIDENCE_PROMPT.md"
    questions_file: str = "/kaggle/input/your-input-dataset/questions_clean_audit copy.jsonl"
    table_mode: str = "txt"  # one of: txt, csv, json, html
    table_dir: str = "/kaggle/input/your-input-dataset/table_inputs_2"

    output_csv: str = "/kaggle/working/qwen14_results.csv"
    limit: int = 0
    qids: str = ""  # comma-separated IDs, e.g. "366,367,370"
    delay_seconds: float = 0.0

    max_new_tokens: int = 768
    do_sample: bool = False
    temperature: float = 1.0
    top_p: float = 1.0
    trust_remote_code: bool = True


CFG = RunConfig()
print("Current configuration:")
for key, value in asdict(CFG).items():
    print(f"  {key}: {value}")


BASE_CSV_COLUMNS = ["id", "question", "correct_answer", "model_answer",
                    "full_response", "full_prompt", "filename", "sub_type", "EM",
                    "strategy", "prompt_tokens", "schema_match"]

MODE_TO_SUFFIX = {
    "txt": "txt",
    "csv": "csv",
    "json": "json",
    "html": "html",
}

SYSTEM_MESSAGE = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."


def _process_decimal(value: str) -> str:
    """Normalize number format without forced rounding."""
    try:
        numeric = float(value)
        if numeric == int(numeric):
            return str(int(numeric))
        return str(numeric)
    except (ValueError, OverflowError):
        return value


def normalize_answer_for_em(value: str) -> str:
    """Normalize free-form text for exact-match style evaluation."""
    normalized = value.strip().strip('"').strip("'").rstrip(".").strip()
    normalized = normalized.replace("$", "").replace("%", "").replace(",", "")
    normalized = re.sub(r"\b(a|an|the)\b", " ", normalized, flags=re.IGNORECASE)
    normalized = re.sub(r"[^\w\s]", " ", normalized)
    normalized = " ".join(normalized.lower().split())
    return _process_decimal(normalized)


def exact_match_with_normalization(predicted: str, label: str) -> int:
    """Return EM=1 if raw match or normalized match succeeds, else 0."""
    if not predicted.strip():
        return 0
    if predicted.strip().lower() == label.strip().lower():
        return 1

    normalized_pred = normalize_answer_for_em(predicted)
    normalized_label = normalize_answer_for_em(label)
    return 1 if normalized_pred == normalized_label else 0


# ---------------------------------------------------------------------------
# Prompt builder (copied from run_experiment_csv.py)
# ---------------------------------------------------------------------------

def load_prompt_template(prompt_file: Path) -> str:
    """Load prompt template from an external .md file."""
    text = prompt_file.read_text(encoding="utf-8")
    # Strip the example table HTML from the template (everything in # 7 between
    # the placeholder markers should already use {placeholders})
    return text


def build_prompt(table_txt: str, question: str, template: str) -> str:
    """Parse the table .txt file and inject into the prompt template."""
    col_start  = table_txt.find("[COLUMN STRUCTURE]")
    row_start  = table_txt.find("[ROW STRUCTURE]")
    html_start = table_txt.find("[TABLE HTML]")

    col_section  = table_txt[col_start:row_start].strip()  if col_start  >= 0 else "[COLUMN STRUCTURE]\n(none)"
    row_section  = table_txt[row_start:html_start].strip() if row_start  >= 0 else "[ROW STRUCTURE]\n(none)"
    html_section = table_txt[html_start:].strip()          if html_start >= 0 else "[TABLE HTML]\n(unavailable)"

    return template.format(
        col_structure_section=col_section,
        row_structure_section=row_section,
        html_section=html_section,
        question=question,
    )


def build_prompt_json(json_path: Path, question: str, template: str) -> str:
    """Load a trees_json/*.json file and inject into the JSON prompt template."""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    json_section = "[TABLE JSON]\n" + json.dumps(data, ensure_ascii=False, indent=2)
    return template.format(
        json_section=json_section,
        question=question,
    )


def clean_html(raw_html: str) -> str:
    """Extract only <table> and any sibling <caption> from an HTML document.

    Strips <head>, <style>, <meta>, scripts, and all body content outside the table.
    Removes inline style attributes from retained elements.
    """
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(raw_html, "html.parser")
    table = soup.find("table")
    if table is None:
        return raw_html  # fallback: return as-is
    caption = soup.find("caption")

    def _strip_inline_styles(tag):
        if tag is None:
            return
        if hasattr(tag, "attrs"):
            tag.attrs.pop("style", None)
        for child in tag.find_all(True):
            child.attrs.pop("style", None)

    _strip_inline_styles(table)
    if caption and caption.find_parent("table") is None:
        _strip_inline_styles(caption)

    parts = []
    if caption and caption.find_parent("table") is None:
        parts.append(str(caption))
    parts.append(str(table))
    return "\n".join(parts)


def build_prompt_html(html_path: Path, question: str, template: str) -> str:
    """Load raw HTML, clean it, and inject into the prompt template."""
    raw = html_path.read_text(encoding="utf-8", errors="ignore")
    cleaned = clean_html(raw)
    html_section = "[TABLE HTML]\n" + cleaned
    return template.format(
        col_structure_section="",
        row_structure_section="",
        html_section=html_section,
        question=question,
    )


def _filename_for_path(path: Path, input_mode: str) -> str:
    """Return filename string with legacy formatting behavior."""
    if input_mode == "txt":
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)
    return str(path)


def _build_prompt_for_mode(input_mode: str, table_path: Path, question: str, template: str) -> str:
    """Build a prompt from table_path using the declared input mode."""
    if input_mode == "html":
        return build_prompt_html(table_path, question, template)
    if input_mode == "json":
        return build_prompt_json(table_path, question, template)

    table_txt = table_path.read_text(encoding="utf-8")
    if input_mode == "csv":
        table_txt = (
            "[COLUMN STRUCTURE]\n(unavailable)\n"
            "[ROW STRUCTURE]\n(unavailable)\n"
            f"[TABLE HTML]\n{table_txt}"
        )
    return build_prompt(table_txt, question, template)


# ---------------------------------------------------------------------------
# Answer extraction & EM (copied from run_experiment_csv.py)
# ---------------------------------------------------------------------------

_FINAL_ANSWER_RE = re.compile(
    r'(?:\*?\*?\[?\s*Final\s+Answer\s*\]?\*?\*?\s*[:\-]\s*)(.*)',
    re.IGNORECASE,
)
_OUTPUT_RE = re.compile(
    r'(?:\*?\*?\#?\s*Output\s*\*?\*?\s*[:\-]\s*)(.*)',
    re.IGNORECASE,
)
_SECTION_HEADER_RE = re.compile(
    r'^\s*(?:\*\*\s*)?\[?\s*(?:step\s*\d+|reasoning|analysis|explanation|notes?|confidence|final\s+answer|output)\b.*$',
    re.IGNORECASE,
)


def _strip_code_fences(text: str) -> str:
    """Remove leading/trailing triple-backtick fences."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r'^```[^\n]*\n?', '', text)
    if text.endswith("```"):
        text = re.sub(r'\n?```$', '', text)
    return text.strip()


def _collect_section(lines: list[str], start_idx: int, first_part: str) -> str:
    collected = [first_part] if first_part else []
    for subsequent in lines[start_idx + 1:]:
        if _SECTION_HEADER_RE.match(subsequent):
            break
        collected.append(subsequent.rstrip())
    return _strip_code_fences("\n".join(collected).strip())


def _find_last_marker(lines: list[str], pattern: re.Pattern[str]) -> int:
    best_idx = -1
    for i, line in enumerate(lines):
        if pattern.search(line):
            best_idx = i
    return best_idx


def _is_meaningful_answer(text: str) -> bool:
    """Require at least one alphanumeric character to avoid punctuation noise."""
    return bool(re.search(r"[A-Za-z0-9]", text or ""))


def extract_answer(response: str) -> str:
    """Extract answer text from full model response.

    Priority:
      1) Last [Final Answer]: section
      2) Last Output: section
      3) Last non-empty line fallback

    Multi-line answers are preserved until a known section header appears.
    """
    lines = response.strip().splitlines()

    final_idx = _find_last_marker(lines, _FINAL_ANSWER_RE)
    if final_idx >= 0:
        match = _FINAL_ANSWER_RE.search(lines[final_idx])
        first_part = match.group(1).strip() if match else ""
        answer = _collect_section(lines, final_idx, first_part)
        if answer and _is_meaningful_answer(answer):
            return answer

    output_idx = _find_last_marker(lines, _OUTPUT_RE)
    if output_idx >= 0:
        match = _OUTPUT_RE.search(lines[output_idx])
        first_part = match.group(1).strip() if match else ""
        answer = _collect_section(lines, output_idx, first_part)
        if answer and _is_meaningful_answer(answer):
            return answer

    # fallback: last non-empty line
    for line in reversed(lines):
        if line.strip() and _is_meaningful_answer(line.strip()):
            return line.strip()
    return ""


def load_questions(path: Path) -> list[dict]:
    qs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                qs.append(json.loads(line))
    return qs


# ---------------------------------------------------------------------------
# Qwen local inference helpers
# ---------------------------------------------------------------------------

def _parse_qids(raw: str) -> set[str]:
    return {q.strip() for q in (raw or "").split(",") if q.strip()}


def _table_path_for_id(table_id: str, table_mode: str, table_dir: Path) -> Path:
    mode = table_mode.lower()
    if mode not in MODE_TO_SUFFIX:
        raise ValueError(f"Unsupported table_mode='{table_mode}'. Use one of: {sorted(MODE_TO_SUFFIX)}")
    return table_dir / f"{table_id}.{MODE_TO_SUFFIX[mode]}"


def load_qwen_model(cfg: RunConfig):
    print(f"Loading model from: {cfg.model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=cfg.trust_remote_code,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        trust_remote_code=cfg.trust_remote_code,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Model and tokenizer loaded.")
    return model, tokenizer


def call_qwen(prompt: str, model, tokenizer, cfg: RunConfig) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt")
    model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}

    generate_kwargs = {
        "max_new_tokens": cfg.max_new_tokens,
        "do_sample": cfg.do_sample,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if cfg.do_sample:
        generate_kwargs["temperature"] = cfg.temperature
        generate_kwargs["top_p"] = cfg.top_p

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, **generate_kwargs)

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs["input_ids"], generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


def preview_first_prompt(cfg: RunConfig, max_chars: int = 3500) -> str:
    prompt_template = load_prompt_template(Path(cfg.prompt_file))
    questions = load_questions(Path(cfg.questions_file))
    if not questions:
        raise ValueError("No questions found in questions_file.")

    q = questions[0]
    table_id = str(q.get("table_id") or q.get("FileName") or "")
    question = str(q.get("query") or q.get("Question") or "")
    if not table_id:
        raise ValueError("First question has no table_id/FileName.")

    table_path = _table_path_for_id(table_id, cfg.table_mode, Path(cfg.table_dir))
    if not table_path.exists():
        raise FileNotFoundError(f"Table file not found: {table_path}")

    prompt = _build_prompt_for_mode(cfg.table_mode.lower(), table_path, question, prompt_template)
    return prompt[:max_chars]


def run_experiment_qwen(cfg: RunConfig, model, tokenizer) -> Path:
    prompt_path = Path(cfg.prompt_file)
    questions_path = Path(cfg.questions_file)
    table_dir = Path(cfg.table_dir)
    output_path = Path(cfg.output_csv)
    input_mode = cfg.table_mode.lower()

    if not prompt_path.exists():
        raise FileNotFoundError(f"Prompt file not found: {prompt_path}")
    if not questions_path.exists():
        raise FileNotFoundError(f"Questions file not found: {questions_path}")
    if not table_dir.exists():
        raise FileNotFoundError(f"Table directory not found: {table_dir}")
    if input_mode not in MODE_TO_SUFFIX:
        raise ValueError(f"Unsupported table_mode='{cfg.table_mode}'. Use one of: {sorted(MODE_TO_SUFFIX)}")

    prompt_template = load_prompt_template(prompt_path)
    questions = load_questions(questions_path)

    qids = _parse_qids(cfg.qids)
    if qids:
        questions = [q for q in questions if str(q.get("id", "")) in qids]
    elif cfg.limit:
        questions = questions[:cfg.limit]

    table_meta = {}
    metadata_path = table_dir / "table_metadata.json"
    if metadata_path.exists():
        with open(metadata_path, encoding="utf-8") as f:
            table_meta = json.load(f)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    already_done = set()
    write_header = True

    if output_path.exists():
        with open(output_path, encoding="utf-8") as f:
            reader = csv.DictReader(f)
            existing_cols = reader.fieldnames or []
            if existing_cols and existing_cols != BASE_CSV_COLUMNS:
                raise RuntimeError(
                    "Output CSV schema mismatch. "
                    f"Expected columns: {BASE_CSV_COLUMNS} | Found: {existing_cols}"
                )
            for row in reader:
                already_done.add(str(row.get("id", "")))
        write_header = False
        print(f"Resuming run - {len(already_done)} rows already present.")

    total = len(questions)
    em_hits = 0
    done = 0
    errors = 0

    with open(output_path, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=BASE_CSV_COLUMNS)
        if write_header:
            writer.writeheader()

        for q in tqdm(questions, total=total, desc="Running Qwen"):
            qid = str(q.get("id", ""))
            if not qid:
                errors += 1
                continue
            if qid in already_done:
                continue

            tid = str(q.get("table_id") or q.get("FileName") or "")
            question = str(q.get("query") or q.get("Question") or "")
            label = str(q.get("label") or q.get("FinalAnswer") or q.get("answer") or "")
            sub_type = str(q.get("sub_type") or q.get("SubQType") or "")

            if not tid:
                print(f"SKIP Q{qid}: table_id/FileName missing")
                errors += 1
                continue

            tbl_path = _table_path_for_id(tid, input_mode, table_dir)
            filename = _filename_for_path(tbl_path, input_mode)
            if not tbl_path.exists():
                print(f"SKIP Q{qid} ({tid}): table file missing -> {tbl_path}")
                errors += 1
                continue

            full_prompt = _build_prompt_for_mode(input_mode, tbl_path, question, prompt_template)

            try:
                t0 = time.perf_counter()
                full_response = call_qwen(full_prompt, model, tokenizer, cfg)
                elapsed = time.perf_counter() - t0

                model_answer = extract_answer(full_response)
                em = exact_match_with_normalization(model_answer, label)
                em_hits += em
                done += 1

                status = "CORRECT" if em else "WRONG"
                pct = em_hits / done * 100 if done else 0
                print(
                    f"Q{qid} {status} [{em_hits}/{done} {pct:.0f}%] "
                    f"({elapsed:.1f}s) pred={model_answer[:60]!r} label={label[:60]!r}"
                )
            except Exception as e:
                full_response = f"ERROR: {e}"
                model_answer = ""
                em = 0
                errors += 1
                done += 1
                print(f"Q{qid} ERROR: {e}")

            row = {
                "id": qid,
                "question": question,
                "correct_answer": label,
                "model_answer": model_answer,
                "full_response": full_response,
                "full_prompt": full_prompt,
                "filename": filename,
                "sub_type": sub_type,
                "EM": em,
                "strategy": table_meta.get(tid, {}).get("strategy", ""),
                "prompt_tokens": len(full_prompt) // 4,
                "schema_match": table_meta.get(tid, {}).get("schema_match", ""),
            }

            writer.writerow(row)
            csvfile.flush()

            if cfg.delay_seconds > 0:
                time.sleep(cfg.delay_seconds)

    acc = em_hits / done * 100 if done else 0
    print("\n" + "=" * 70)
    print(f"DONE | {em_hits}/{done} correct | EM = {acc:.1f}%")
    if errors:
        print(f"Errors/skips: {errors}")
    print(f"Results saved to: {output_path}")
    return output_path

In [ ]:
# Byte-identical override from run_experiment_csv.py
PROJECT_ROOT = Path.cwd()


def _filename_for_path(path: Path, input_mode: str) -> str:
    """Return filename string with legacy formatting behavior."""
    if input_mode == "txt":
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)
    return str(path)

In [ ]:
# Load Qwen model + tokenizer (this can take a few minutes)
model, tokenizer = load_qwen_model(CFG)

In [ ]:
# Preview one fully rendered prompt before the full run
preview_text = preview_first_prompt(CFG, max_chars=5000)
print(preview_text)
print("\n[Preview truncated to 5000 chars]")

In [ ]:
# Run full experiment and write incremental CSV output
output_csv_path = run_experiment_qwen(CFG, model, tokenizer)

# Quick view of generated rows
results_df = pd.read_csv(output_csv_path)
print(f"Rows written: {len(results_df)}")
display(results_df.tail(10))

## Kaggle Run Checklist

- [ ] Add your datasets to Kaggle Input: model, prompt file, questions jsonl, table directory.
- [ ] In `CFG`, set `model_name`, `prompt_file`, `questions_file`, and `table_dir`.
- [ ] Set `table_mode` to match your table files: `txt`, `csv`, `json`, or `html`.
- [ ] Optional: set `limit` or `qids` for a small smoke test first.
- [ ] Run the model loading cell.
- [ ] Run the preview cell and verify prompt formatting.
- [ ] Run the full experiment cell.
- [ ] Download `/kaggle/working/qwen14_results.csv` after completion.
- [ ] If interrupted, rerun the final cell: it resumes from existing rows in the same CSV.